# LeetCode 1089 — Duplicate Zeros bằng Python

**Link đề:** https://leetcode.com/problems/duplicate-zeros/
**Độ khó:** Easy · **Chủ đề:** Array, Two Pointers

```python
class Solution:
    def duplicateZeros(self, arr: list[int]) -> None:
        """
        Do not return anything, modify arr in-place instead.
        """
```

## Mục tiêu học
1. Hiểu **sửa mảng tại chỗ** (in-place) nghĩa là gì, và vì sao hàm **không return** mà vẫn "trả kết quả".
2. Phân biệt gán lại biến `arr = ...` với sửa nội dung `arr[:] = ...` / `arr[i] = ...`.
3. Viết 3 lời giải: chèn + bỏ cuối (`insert`/`pop`, `O(n²)`) → mảng phụ (`O(n)` bộ nhớ) → **hai con trỏ ghi từ cuối** (`O(1)` bộ nhớ).
4. Hiểu nút thắt "ghi đè dữ liệu chưa đọc" và mẹo **duyệt ngược**.
5. Trace kỹ ca biên (số 0 cuối cùng chỉ còn chỗ cho **một** bản sao), chứng minh, Time/Space, bài tập.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — nhân đôi mỗi số 0, giữ nguyên độ dài mảng

Cho mảng số nguyên **độ dài cố định** `arr`. **Nhân đôi** mỗi lần xuất hiện của số `0` và **dịch** các phần tử còn lại sang phải. Phần tử nào bị đẩy **vượt quá độ dài ban đầu** thì **bỏ đi**.

Sửa **trực tiếp trên `arr`** (in-place) và **không trả về** gì.

| `arr` trước | `arr` sau | Giải thích |
|---|---|---|
| `[1,0,2,3,0,4,5,0]` | `[1,0,0,2,3,0,0,4]` | hai số 0 đầu được nhân đôi; `5` và `0` cuối bị đẩy ra ngoài |
| `[1,2,3]` | `[1,2,3]` | không có số 0 |

**Constraints:**
- `1 <= arr.length <= 10^4`
- `0 <= arr[i] <= 9`

In [1]:
arr = [1, 0, 2, 3, 0, 4, 5, 0]
# Hình dung: viết ra mảng "dài vô hạn" rồi cắt đúng len(arr)
long_version = []
for x in arr:
    long_version.append(x)
    if x == 0:
        long_version.append(0)
print("dài đầy đủ:", long_version)
print("cắt còn   :", long_version[:len(arr)])

dài đầy đủ: [1, 0, 0, 2, 3, 0, 0, 4, 5, 0, 0]
cắt còn   : [1, 0, 0, 2, 3, 0, 0, 4]


## 2. Nền tảng Python cần biết

### 2.1. List là đối tượng **có thể thay đổi** (mutable) — hàm sửa được list của người gọi

Khi gọi `f(arr)`, hàm nhận **cùng một list** (không phải bản sao). Sửa `arr[i] = ...` bên trong hàm thì bên ngoài **thấy được**. Đó là lý do LeetCode không cần `return`: họ kiểm tra chính list đã đưa vào.

In [2]:
def set_first_to_99(a):
    a[0] = 99          # sửa nội dung -> người gọi thấy

nums = [1, 2, 3]
set_first_to_99(nums)
print(nums)            # [99, 2, 3]

[99, 2, 3]


### 2.2. Bẫy: `arr = ...` bên trong hàm **không** sửa list gốc

`arr = new_list` chỉ khiến **tên** `arr` trong hàm trỏ sang list khác; list gốc không đổi. Muốn thay toàn bộ nội dung, dùng **gán lát cắt** `arr[:] = new_list`.

In [3]:
def rebind(a):
    a = [7, 7, 7]          # chỉ đổi tên cục bộ

def replace_content(a):
    a[:] = [7, 7, 7]       # thay nội dung của chính list đó

x = [1, 2, 3]; rebind(x);          print("sau rebind        :", x)
y = [1, 2, 3]; replace_content(y); print("sau replace_content:", y)

sau rebind        : [1, 2, 3]
sau replace_content: [7, 7, 7]


### 2.3. `list.insert(i, x)` và `list.pop()`

- `a.insert(i, x)` chèn `x` vào vị trí `i`, **dịch mọi phần tử sau** sang phải → `O(n)`.
- `a.pop()` bỏ phần tử **cuối** → `O(1)`. (`a.pop(i)` bỏ ở vị trí `i` → `O(n)`.)

In [4]:
a = [1, 0, 2, 3]
a.insert(2, 0)       # chèn 0 vào vị trí 2
print(a)             # [1, 0, 0, 2, 3]
a.pop()              # bỏ phần tử cuối để giữ độ dài
print(a)             # [1, 0, 0, 2]

[1, 0, 0, 2, 3]
[1, 0, 0, 2]


### 2.4. `range` đếm lùi và `list.count`

- `range(n - 1, -1, -1)` cho `n-1, n-2, ..., 0`.
- `arr.count(0)` đếm số lần xuất hiện của `0`.

In [5]:
print(list(range(4, -1, -1)))
print([1, 0, 2, 3, 0, 4, 5, 0].count(0))

[4, 3, 2, 1, 0]
3


## 3. Cách 1 — Trực giác: gặp 0 thì chèn thêm 0, bỏ phần tử cuối

Đi từ trái sang phải với chỉ số `i`:
- Nếu `arr[i] == 0`: `arr.insert(i + 1, 0)` rồi `arr.pop()` (giữ độ dài), và **nhảy qua** số 0 vừa chèn: `i += 2`.
- Ngược lại: `i += 1`.

Phải dùng `while` (không dùng `for i in range(n)`) vì ta tự điều khiển bước nhảy.

In [6]:
def duplicate_zeros_insert(arr: list[int]) -> None:
    n = len(arr)
    i = 0
    while i < n:
        if arr[i] == 0:
            arr.insert(i + 1, 0)
            arr.pop()
            i += 2          # bỏ qua số 0 vừa chèn
        else:
            i += 1

a = [1, 0, 2, 3, 0, 4, 5, 0]
duplicate_zeros_insert(a)
print(a)
b2 = [1, 2, 3]
duplicate_zeros_insert(b2)
print(b2)

[1, 0, 0, 2, 3, 0, 0, 4]
[1, 2, 3]


### Vì sao phải `i += 2`?

Nếu chỉ `i += 1`, vòng sau gặp lại **chính số 0 vừa chèn** → lại chèn → dây chuyền: mọi phần tử phía sau biến thành 0!

In [7]:
def wrong_step_one(arr):
    n = len(arr)
    i = 0
    while i < n:
        if arr[i] == 0:
            arr.insert(i + 1, 0)
            arr.pop()
        i += 1                   # sai: gặp lại 0 vừa chèn

a = [1, 0, 2, 3]
wrong_step_one(a)
print("sai :", a, "| đúng: [1, 0, 0, 2]")

sai : [1, 0, 0, 0] | đúng: [1, 0, 0, 2]


## 4. Cách 2 — Dùng mảng phụ rồi chép ngược vào `arr`

Cách dễ đúng nhất: xây danh sách mới `res`, gặp 0 thì thêm hai số 0; dừng khi `res` đủ `n` phần tử. Cuối cùng `arr[:] = res`.

Gợi ý của đề: *"in-place không có nghĩa là không được dùng mảng khác"* — cách này hợp lệ, chỉ tốn thêm `O(n)` bộ nhớ.

In [8]:
def duplicate_zeros_copy(arr: list[int]) -> None:
    n = len(arr)
    res = []
    for x in arr:
        if len(res) == n:
            break
        res.append(x)
        if x == 0 and len(res) < n:     # còn chỗ thì nhân đôi
            res.append(0)
    arr[:] = res                        # sửa nội dung list gốc

a = [1, 0, 2, 3, 0, 4, 5, 0]
duplicate_zeros_copy(a)
print(a)

[1, 0, 0, 2, 3, 0, 0, 4]


## 5. Nút thắt & insight: làm sao `O(n)` thời gian **và** `O(1)` bộ nhớ?

- Cách 1: mỗi lần `insert` dịch cả đuôi mảng → `O(n)` một lần, tổng `O(n²)` (mảng toàn 0).
- Cách 2: nhanh nhưng tốn mảng phụ.

**Vấn đề khi ghi đè tại chỗ từ trái sang phải:** nếu ghi `0` vào vị trí `i + 1`, ta **phá mất** giá trị cũ `arr[i+1]` chưa kịp đọc.

**Insight (gợi ý của đề):** biết trước vị trí đích của mọi phần tử! Phần tử ở chỉ số `i` sẽ chuyển tới `i + (số số 0 đứng trước nó)`. Đích luôn **≥** nguồn ⇒ nếu **ghi từ cuối về đầu**, chỗ ta ghi vào là chỗ **đã đọc xong** hoặc nằm ngoài mảng → **không phá dữ liệu chưa đọc**.

Đây là mẹo kinh điển "**dịch sang phải thì duyệt ngược**" (giống `memmove`, hay bài Merge Sorted Array).

In [9]:
arr = [1, 0, 2, 3, 0, 4, 5, 0]
zeros_before = 0
for i, x in enumerate(arr):
    dest = i + zeros_before
    print(f"i={i}, x={x}, số 0 đứng trước={zeros_before}, vị trí đích={dest}" + ("  (ra ngoài)" if dest >= len(arr) else ""))
    if x == 0:
        zeros_before += 1

i=0, x=1, số 0 đứng trước=0, vị trí đích=0
i=1, x=0, số 0 đứng trước=0, vị trí đích=1
i=2, x=2, số 0 đứng trước=1, vị trí đích=3
i=3, x=3, số 0 đứng trước=1, vị trí đích=4
i=4, x=0, số 0 đứng trước=1, vị trí đích=5
i=5, x=4, số 0 đứng trước=2, vị trí đích=7
i=6, x=5, số 0 đứng trước=2, vị trí đích=8  (ra ngoài)
i=7, x=0, số 0 đứng trước=2, vị trí đích=9  (ra ngoài)


Nhìn bảng: `5` (i=6) có đích 8 và `0` (i=7) có đích 9 — đều **ngoài** mảng dài 8 → bị bỏ. Số 0 ở i=4 có đích 5, **bản sao** của nó ở 6 — vẫn trong mảng.

## 6. Cách 3 (lời giải chính) — Hai con trỏ, ghi từ cuối về đầu

**Đoạn 1 — đếm số 0:** `zeros = arr.count(0)`. Mảng "dài đầy đủ" có độ dài `n + zeros`.

**Đoạn 2 — hai con trỏ:** `i` đọc từ `n - 1` lùi về 0 (nguồn); `j = n + zeros - 1` là vị trí ghi trong mảng **ảo** dài đầy đủ.

**Đoạn 3 — ghi:** với mỗi `i` (lùi dần):
- Nếu `j < n`: `arr[j] = arr[i]` (chỉ ghi khi đích nằm trong mảng thật).
- Nếu `arr[i] == 0`: thêm bản sao: `j -= 1`, và nếu `j < n` thì `arr[j] = 0`.
- `j -= 1`.

Không cần mảng phụ: chỗ ngoài mảng chỉ là **ảo** (ta bỏ qua việc ghi).

In [10]:
def duplicate_zeros_two_pointers(arr: list[int]) -> None:
    n = len(arr)
    zeros = arr.count(0)                 # Đoạn 1
    j = n + zeros - 1                    # Đoạn 2: vị trí ghi trong mảng ảo
    for i in range(n - 1, -1, -1):
        if j < n:                        # Đoạn 3: ghi bản gốc
            arr[j] = arr[i]
        if arr[i] == 0:
            j -= 1
            if j < n:                    # ghi bản sao của số 0
                arr[j] = 0
        j -= 1

a = [1, 0, 2, 3, 0, 4, 5, 0]
duplicate_zeros_two_pointers(a)
print(a)
for t in [[1, 2, 3], [0, 0, 0], [8, 4, 5, 0, 0, 0, 0, 7], [1, 0]]:
    duplicate_zeros_two_pointers(t)
    print(t)

[1, 0, 0, 2, 3, 0, 0, 4]
[1, 2, 3]
[0, 0, 0]
[8, 4, 5, 0, 0, 0, 0, 0]
[1, 0]


### Vì sao `if arr[i] == 0` đọc `arr[i]` **sau** khi có thể đã ghi `arr[j]`?

Vì `j >= i` luôn đúng (đích ≥ nguồn). Nếu `j == i` thì ta vừa ghi `arr[i] = arr[i]` (không đổi). Nếu `j > i` thì không đụng tới `arr[i]`. Nên `arr[i]` vẫn là giá trị gốc.

## 7. Trace cách 3 với `[1,0,2,3,0,4,5,0]`

`n = 8`, `zeros = 3` → `j` bắt đầu ở `10`. Ô ngoài mảng (`j >= 8`) chỉ "ghi ảo".

| `i` | `arr[i]` | `j` ghi bản gốc | Ghi thật? | Bản sao 0 ở `j` | `j` sau |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 7 | 0 | 10 | không | 9 (không) | 8 |
| 6 | 5 | 8 | không | — | 7 |
| 5 | 4 | 7 | `arr[7]=4` | — | 6 |
| 4 | 0 | 6 | `arr[6]=0` | 5 → `arr[5]=0` | 4 |
| 3 | 3 | 4 | `arr[4]=3` | — | 3 |
| 2 | 2 | 3 | `arr[3]=2` | — | 2 |
| 1 | 0 | 2 | `arr[2]=0` | 1 → `arr[1]=0` | 0 |
| 0 | 1 | 0 | `arr[0]=1` | — | -1 |

Kết quả `[1,0,0,2,3,0,0,4]`. Để ý `j - i` giảm dần đúng bằng số số 0 đứng trước, và cuối cùng `j = i - 1 = -1`.

In [11]:
def trace_duplicate_zeros(arr: list[int]) -> None:
    n = len(arr)
    zeros = arr.count(0)
    j = n + zeros - 1
    print(f"n={n}, zeros={zeros}, j bắt đầu={j}")
    for i in range(n - 1, -1, -1):
        note = []
        if j < n:
            arr[j] = arr[i]
            note.append(f"arr[{j}]={arr[i]}")
        else:
            note.append(f"(ảo j={j})")
        if arr[i] == 0:
            j -= 1
            if j < n:
                arr[j] = 0
                note.append(f"sao 0 -> arr[{j}]=0")
            else:
                note.append(f"sao 0 (ảo j={j})")
        j -= 1
        print(f"i={i}, arr[i]={arr[i]}: {', '.join(note):28} | arr={arr}")

trace_duplicate_zeros([1, 0, 2, 3, 0, 4, 5, 0])

n=8, zeros=3, j bắt đầu=10
i=7, arr[i]=0: (ảo j=10), sao 0 (ảo j=9)    | arr=[1, 0, 2, 3, 0, 4, 5, 0]
i=6, arr[i]=5: (ảo j=8)                     | arr=[1, 0, 2, 3, 0, 4, 5, 0]
i=5, arr[i]=4: arr[7]=4                     | arr=[1, 0, 2, 3, 0, 4, 5, 4]
i=4, arr[i]=0: arr[6]=0, sao 0 -> arr[5]=0  | arr=[1, 0, 2, 3, 0, 0, 0, 4]
i=3, arr[i]=3: arr[4]=3                     | arr=[1, 0, 2, 3, 3, 0, 0, 4]
i=2, arr[i]=2: arr[3]=2                     | arr=[1, 0, 2, 2, 3, 0, 0, 4]
i=1, arr[i]=0: arr[2]=0, sao 0 -> arr[1]=0  | arr=[1, 0, 0, 2, 3, 0, 0, 4]
i=0, arr[i]=1: arr[0]=1                     | arr=[1, 0, 0, 2, 3, 0, 0, 4]


### Ca biên quan trọng: số 0 chỉ vừa **một** chỗ ở cuối

`arr = [8,4,5,0,0,0,0,7]` (`n = 8`): số 0 ở `i=5` có 2 số 0 đứng trước nên chiếm hai ô `7` và `8` trong mảng đầy đủ — ô `8` nằm **ngoài**, chỉ ô `7` nằm trong. Trong code, lần ghi đầu (ô `j = 8`) bị bỏ qua, lần ghi bản sao (ô `j = 7`) được thực hiện. Kết quả `[8,4,5,0,0,0,0,0]`. Code xử lý tự nhiên nhờ kiểm tra `j < n` **riêng** cho từng lần ghi.

In [12]:
trace_duplicate_zeros([8, 4, 5, 0, 0, 0, 0, 7])

n=8, zeros=4, j bắt đầu=11
i=7, arr[i]=7: (ảo j=11)                    | arr=[8, 4, 5, 0, 0, 0, 0, 7]
i=6, arr[i]=0: (ảo j=10), sao 0 (ảo j=9)    | arr=[8, 4, 5, 0, 0, 0, 0, 7]
i=5, arr[i]=0: (ảo j=8), sao 0 -> arr[7]=0  | arr=[8, 4, 5, 0, 0, 0, 0, 0]
i=4, arr[i]=0: arr[6]=0, sao 0 -> arr[5]=0  | arr=[8, 4, 5, 0, 0, 0, 0, 0]
i=3, arr[i]=0: arr[4]=0, sao 0 -> arr[3]=0  | arr=[8, 4, 5, 0, 0, 0, 0, 0]
i=2, arr[i]=5: arr[2]=5                     | arr=[8, 4, 5, 0, 0, 0, 0, 0]
i=1, arr[i]=4: arr[1]=4                     | arr=[8, 4, 5, 0, 0, 0, 0, 0]
i=0, arr[i]=8: arr[0]=8                     | arr=[8, 4, 5, 0, 0, 0, 0, 0]


## 8. Code nộp LeetCode

### Bản hai con trỏ `O(1)` bộ nhớ (lời giải chính)

In [13]:
class Solution:
    def duplicateZeros(self, arr: list[int]) -> None:
        """
        Do not return anything, modify arr in-place instead.
        """
        n = len(arr)
        j = n + arr.count(0) - 1
        for i in range(n - 1, -1, -1):
            if j < n:
                arr[j] = arr[i]
            if arr[i] == 0:
                j -= 1
                if j < n:
                    arr[j] = 0
            j -= 1

for inp, exp in [([1, 0, 2, 3, 0, 4, 5, 0], [1, 0, 0, 2, 3, 0, 0, 4]), ([1, 2, 3], [1, 2, 3])]:
    ret = Solution().duplicateZeros(inp)
    assert ret is None and inp == exp, (inp, exp)
print("Solution (two pointers) PASS")

Solution (two pointers) PASS


### Bản mảng phụ (dễ viết đúng nhất)

In [14]:
class SolutionCopy:
    def duplicateZeros(self, arr: list[int]) -> None:
        """
        Do not return anything, modify arr in-place instead.
        """
        res = []
        for x in arr:
            res.append(x)
            if x == 0:
                res.append(0)
        arr[:] = res[:len(arr)]

a = [1, 0, 2, 3, 0, 4, 5, 0]
SolutionCopy().duplicateZeros(a)
assert a == [1, 0, 0, 2, 3, 0, 0, 4]
print("SolutionCopy PASS")

SolutionCopy PASS


## 9. Vì sao cách 3 đúng?

Gọi `full` là mảng "dài đầy đủ" (mỗi 0 thành hai 0), độ dài `n + zeros`. Kết quả đúng là `full[:n]`.

**Ánh xạ vị trí:** phần tử gốc ở `i` nằm ở `full[i + z_i]`, với `z_i` = số số 0 ở **trước** `i`; nếu `arr[i] == 0` thì bản sao nằm ở `full[i + z_i + 1]`.

**Invariant:** khi bắt đầu xử lý `i` (đi lùi), `j = i + z_i + (1 nếu arr[i] == 0 else 0)` — tức `j` là vị trí **cuối cùng** mà phần tử `i` chiếm trong `full`.
- Lúc đầu `i = n-1`: `j = n + zeros - 1` = vị trí cuối của `full`. Đúng.
- Mỗi bước, ta ghi phần tử `i` vào đúng (các) vị trí của nó trong `full` (chỉ khi `< n`), rồi lùi `j` qua đúng số ô nó chiếm (1 hoặc 2) → `j` thành vị trí cuối của phần tử `i - 1`. Invariant giữ.

**Không phá dữ liệu chưa đọc:** mọi ô được ghi có chỉ số `>= i` (vì `z_i >= 0`), còn các phần tử chưa đọc có chỉ số `< i`. ⇒ an toàn.

## 10. Độ phức tạp

Gọi `n = len(arr)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. `insert` + `pop` | `O(n²)` worst (nhiều số 0) | `O(1)` | mỗi `insert` dịch đuôi `O(n)` |
| 2. Mảng phụ | `O(n)` | `O(n)` | dễ đúng nhất |
| 3. Hai con trỏ từ cuối | `O(n)` (2 lượt: đếm + ghi) | `O(1)` | tối ưu |

Đo nhanh trên mảng toàn số 0 để thấy `O(n²)`:

In [15]:
import time
for n in [2000, 4000, 8000]:
    a = [0] * n
    t0 = time.perf_counter(); duplicate_zeros_insert(a); t1 = time.perf_counter()
    c = [0] * n
    t2 = time.perf_counter(); duplicate_zeros_two_pointers(c); t3 = time.perf_counter()
    print(f"n={n}: insert {t1 - t0:.4f}s | two pointers {t3 - t2:.5f}s")

n=2000: insert 0.0003s | two pointers 0.00021s
n=4000: insert 0.0009s | two pointers 0.00035s
n=8000: insert 0.0036s | two pointers 0.00094s


(Thời gian cụ thể tùy máy; quan sát chính: cách `insert` tăng khoảng **4 lần** khi `n` gấp đôi, cách hai con trỏ chỉ khoảng **2 lần**.)

## 11. Bộ test

Nhóm test:
1. Hai ví dụ đề.
2. Một phần tử: `[0]`, `[5]`.
3. Toàn 0: không đổi.
4. Số 0 ở cuối cùng: `[1, 0]` → `[1, 0]`.
5. Số 0 chỉ còn chỗ cho một bản: `[8,4,5,0,0,0,0,7]`.
6. Kiểm tra hàm **trả `None`** và sửa **đúng list gốc**.
7. Đối chiếu ngẫu nhiên 3 cách + một bản "định nghĩa" dùng mảng `full`.

In [16]:
def reference(arr):
    full = []
    for x in arr:
        full += [0, 0] if x == 0 else [x]
    return full[:len(arr)]

def check(name, inp, expected, fn):
    a = inp[:]
    ret = fn(a)
    assert ret is None, (name, "phải trả None")
    assert a == expected, (name, a, expected)
    print(f"{name:26} PASS -> {a}")

tests = [
    ("đề 1", [1, 0, 2, 3, 0, 4, 5, 0], [1, 0, 0, 2, 3, 0, 0, 4]),
    ("đề 2", [1, 2, 3], [1, 2, 3]),
    ("[0]", [0], [0]),
    ("[5]", [5], [5]),
    ("toàn 0", [0, 0, 0, 0], [0, 0, 0, 0]),
    ("0 ở cuối", [1, 0], [1, 0]),
    ("0 vừa một chỗ", [8, 4, 5, 0, 0, 0, 0, 7], [8, 4, 5, 0, 0, 0, 0, 0]),
    ("0 đầu mảng", [0, 1, 2], [0, 0, 1]),
]
for name, inp, exp in tests:
    assert reference(inp) == exp
    check(name, inp, exp, duplicate_zeros_two_pointers)

đề 1                       PASS -> [1, 0, 0, 2, 3, 0, 0, 4]
đề 2                       PASS -> [1, 2, 3]
[0]                        PASS -> [0]
[5]                        PASS -> [5]
toàn 0                     PASS -> [0, 0, 0, 0]
0 ở cuối                   PASS -> [1, 0]
0 vừa một chỗ              PASS -> [8, 4, 5, 0, 0, 0, 0, 0]
0 đầu mảng                 PASS -> [0, 0, 1]


In [17]:
import random
random.seed(1089)
for _ in range(3000):
    arr = [random.choice([0, 0, 1, 2, 3]) for _ in range(random.randint(1, 15))]
    exp = reference(arr)
    for fn in (duplicate_zeros_insert, duplicate_zeros_copy, duplicate_zeros_two_pointers,
               Solution().duplicateZeros, SolutionCopy().duplicateZeros):
        a = arr[:]
        fn(a)
        assert a == exp, (fn.__name__, arr, a, exp)
print("3000 test ngẫu nhiên: 5 cách khớp với định nghĩa")

3000 test ngẫu nhiên: 5 cách khớp với định nghĩa


## 12. Lỗi người mới hay gặp

### Lỗi 1: Ghi đè từ trái sang phải → mất dữ liệu

In [18]:
def wrong_left_to_right(arr):
    n = len(arr)
    i = 0
    while i < n - 1:
        if arr[i] == 0:
            arr[i + 1] = 0      # phá mất giá trị cũ ở i+1!
            i += 2
        else:
            i += 1

a = [1, 0, 2, 3, 0, 4, 5, 0]
wrong_left_to_right(a)
print("sai :", a)
print("đúng:", reference([1, 0, 2, 3, 0, 4, 5, 0]))

sai : [1, 0, 0, 3, 0, 0, 5, 0]
đúng: [1, 0, 0, 2, 3, 0, 0, 4]


### Lỗi 2: `arr = res` thay vì `arr[:] = res`

Bên trong hàm, `arr = res` chỉ đổi tên cục bộ → LeetCode thấy mảng **không đổi**.

In [19]:
def wrong_rebind(arr):
    res = reference(arr)
    arr = res                  # không sửa list gốc

a = [1, 0, 2, 3]
wrong_rebind(a)
print("sai :", a, "| đúng:", reference([1, 0, 2, 3]))

sai : [1, 0, 2, 3] | đúng: [1, 0, 0, 2]


### Lỗi 3: `return arr` và nghĩ thế là xong

LeetCode bỏ qua giá trị trả về; chỉ kiểm tra `arr` gốc. Nếu bạn tạo mảng mới rồi `return` thì vẫn sai.

### Lỗi 4: Ghi bản gốc và bản sao bằng **cùng một** kiểm tra biên

```python
if j < n:
    arr[j] = arr[i]
    if arr[i] == 0:
        arr[j - 1] = 0      # sai khi bản gốc ngoài mảng nhưng bản sao trong mảng (hoặc ngược lại)
```

Với `[8,4,5,0,0,0,0,7]`, số 0 ở `i=5` có ô đầu tiên được ghi (`j = 8`) nằm ngoài mảng nhưng ô bản sao (`j = 7`) nằm trong. Kiểm tra gộp sẽ bỏ sót ô 7 (để lại số `7` cũ) → phải kiểm tra `j < n` **riêng cho từng lần ghi**.

### Lỗi 5: Quên `i += 2` trong cách `insert` → mọi thứ thành 0 (đã thấy ở mục 3).

## 13. Mẹo nhận dạng pattern

- "**Sửa tại chỗ**" + "**dịch sang phải / chèn thêm**" → nghĩ tới **ghi từ cuối về đầu** với hai con trỏ (nguồn và đích).
- Biết trước kích thước kết quả (`n + số 0`) → đặt con trỏ đích ở cuối "mảng ảo".
- "Xóa / nén sang trái" thì ngược lại: **ghi từ đầu**, con trỏ ghi đi chậm hơn con trỏ đọc.

| Bài liên quan | Hướng ghi |
|---|---|
| 88. Merge Sorted Array | từ cuối (đích ở cuối nums1) |
| 27. Remove Element / 26. Remove Duplicates | từ đầu (nén trái) |
| 283. Move Zeroes | từ đầu |
| 1470. Shuffle the Array | chỉ số đích tính trước |

## 14. Biến thể

### Biến thể A — Nhân đôi một giá trị bất kỳ `v` (không chỉ 0)

In [20]:
def duplicate_value(arr: list[int], v: int) -> None:
    n = len(arr)
    j = n + arr.count(v) - 1
    for i in range(n - 1, -1, -1):
        if j < n:
            arr[j] = arr[i]
        if arr[i] == v:
            j -= 1
            if j < n:
                arr[j] = v
        j -= 1

a = [1, 2, 3, 2, 5]
duplicate_value(a, 2)
print(a)     # [1, 2, 2, 3, 2]

[1, 2, 2, 3, 2]


### Biến thể B — Ngược lại: **xóa** mọi số 0 và dồn sang trái, điền 0 ở cuối (Move Zeroes)

In [21]:
def move_zeroes(arr: list[int]) -> None:
    w = 0                                   # con trỏ ghi
    for x in arr:
        if x != 0:
            arr[w] = x
            w += 1
    for k in range(w, len(arr)):
        arr[k] = 0

a = [0, 1, 0, 3, 12]
move_zeroes(a)
print(a)     # [1, 3, 12, 0, 0]

[1, 3, 12, 0, 0]


## 15. Bài tập tự luyện (tăng dần)

**Bài 1 (tính tay):** `arr = [0, 4, 1, 0, 0, 8, 0, 0, 3]`. Viết kết quả sau `duplicateZeros`.

**Bài 2 (dự đoán output):** sau đoạn code dưới, `nums` là gì?
```python
def f(a):
    a = a + [0]
    a[0] = 9
nums = [1, 2]
f(nums)
```

**Bài 3 (điền chỗ trống):** hoàn thiện `ex_duplicate_zeros` (hai con trỏ từ cuối).

**Bài 4 (biến thể):** `ex_triplicate_zeros(arr)` — mỗi số 0 thành **ba** số 0, vẫn giữ độ dài, sửa tại chỗ.

**Bài 5 (mini challenge — LeetCode 88):** `ex_merge(nums1, m, nums2, n)` — `nums1` dài `m + n`, `m` phần tử đầu đã sắp, `nums2` đã sắp; trộn vào `nums1` tại chỗ, ghi từ cuối.

In [22]:
def ex_duplicate_zeros(arr: list[int]) -> None:
    n = len(arr)
    # TODO 1: j = vị trí cuối của mảng ảo
    for i in range(n - 1, -1, -1):
        # TODO 2: ghi bản gốc nếu j < n
        # TODO 3: nếu arr[i] == 0: lùi j, ghi bản sao nếu j < n
        # TODO 4: lùi j
        pass

# a = [1, 0, 2, 3, 0, 4, 5, 0]; ex_duplicate_zeros(a); assert a == [1, 0, 0, 2, 3, 0, 0, 4]

In [23]:
def ex_triplicate_zeros(arr: list[int]) -> None:
    # TODO: giống bài 3 nhưng mỗi số 0 chiếm 3 ô
    pass

# a = [1, 0, 2, 3]; ex_triplicate_zeros(a); assert a == [1, 0, 0, 0]
# a = [0, 5, 6, 7, 8]; ex_triplicate_zeros(a); assert a == [0, 0, 0, 5, 6]

In [24]:
def ex_merge(nums1: list[int], m: int, nums2: list[int], n: int) -> None:
    # TODO: i = m - 1, j = n - 1, k = m + n - 1; so sánh và ghi phần tử lớn hơn vào nums1[k]
    pass

# a = [1, 2, 3, 0, 0, 0]; ex_merge(a, 3, [2, 5, 6], 3); assert a == [1, 2, 2, 3, 5, 6]

## 16. Gợi ý 3 mức

**Bài 1:** (1) Viết mảng đầy đủ rồi cắt 9 phần tử. (2) Đầy đủ: `0,0,4,1,0,0,0,0,8,0,0,0,0,3` (mỗi 0 → 0,0). (3) Lấy 9 phần tử đầu.

**Bài 2:** (1) `a + [0]` tạo **list mới**. (2) `a = ...` gán lại tên cục bộ. (3) `a[0] = 9` sửa list mới, không phải `nums`.

**Bài 3:** (1) Như mục 6. (2) `j = n + arr.count(0) - 1`. (3) Hai lần kiểm tra `j < n` riêng biệt.

**Bài 4:** (1) Mảng ảo dài `n + 2·zeros`. (2) Số 0 cần ghi thêm 2 bản sao, mỗi bản lùi `j` một ô. (3) Dùng vòng `for _ in range(2)` cho bản sao.

**Bài 5:** (1) Ghi từ cuối để không đè dữ liệu chưa đọc của `nums1`. (2) Ghi phần tử lớn hơn trong `nums1[i]`, `nums2[j]`. (3) Khi `nums2` hết thì dừng; nếu `nums1` hết trước, chép phần còn lại của `nums2`.

## 17. Đáp án tham khảo

Tự làm trước rồi mới xem. Bài 1: `[0,0,4,1,0,0,0,0,8]`. Bài 2: `nums` vẫn là `[1, 2]`.

In [25]:
def ans_duplicate_zeros(arr):
    n = len(arr)
    j = n + arr.count(0) - 1
    for i in range(n - 1, -1, -1):
        if j < n:
            arr[j] = arr[i]
        if arr[i] == 0:
            j -= 1
            if j < n:
                arr[j] = 0
        j -= 1

def ans_triplicate_zeros(arr):
    n = len(arr)
    j = n + 2 * arr.count(0) - 1
    for i in range(n - 1, -1, -1):
        if j < n:
            arr[j] = arr[i]
        if arr[i] == 0:
            for _ in range(2):
                j -= 1
                if j < n:
                    arr[j] = 0
        j -= 1

def ans_merge(nums1, m, nums2, n):
    i, j, k = m - 1, n - 1, m + n - 1
    while j >= 0:
        if i >= 0 and nums1[i] > nums2[j]:
            nums1[k] = nums1[i]; i -= 1
        else:
            nums1[k] = nums2[j]; j -= 1
        k -= 1

a = [0, 4, 1, 0, 0, 8, 0, 0, 3]; ans_duplicate_zeros(a); assert a == [0, 0, 4, 1, 0, 0, 0, 0, 8]
def f(a):
    a = a + [0]
    a[0] = 9
nums = [1, 2]; f(nums); assert nums == [1, 2]
a = [1, 0, 2, 3, 0, 4, 5, 0]; ans_duplicate_zeros(a); assert a == [1, 0, 0, 2, 3, 0, 0, 4]
a = [1, 0, 2, 3]; ans_triplicate_zeros(a); assert a == [1, 0, 0, 0]
a = [0, 5, 6, 7, 8]; ans_triplicate_zeros(a); assert a == [0, 0, 0, 5, 6]
a = [1, 2, 3, 0, 0, 0]; ans_merge(a, 3, [2, 5, 6], 3); assert a == [1, 2, 2, 3, 5, 6]
a = [0]; ans_merge(a, 0, [1], 1); assert a == [1]
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def duplicateZeros(self, arr: list[int]) -> None:
        n = len(arr)
        j = n + arr.count(0) - 1          # đích cuối trong mảng "ảo"
        for i in range(n - 1, -1, -1):    # đọc từ cuối
            if j < n:
                arr[j] = arr[i]           # bản gốc
            if arr[i] == 0:
                j -= 1
                if j < n:
                    arr[j] = 0            # bản sao
            j -= 1
```

**Câu thần chú:** *"Dịch sang phải tại chỗ → ghi từ cuối."*

**Checklist:**
- [ ] Sửa **tại chỗ**, không `return`, không `arr = ...` (dùng `arr[:] = ...` nếu thay toàn bộ).
- [ ] Không ghi đè dữ liệu chưa đọc (duyệt ngược, hoặc dùng mảng phụ).
- [ ] Kiểm tra `j < n` **riêng** cho bản gốc và bản sao.
- [ ] Cách `insert`: nhớ `pop()` giữ độ dài và `i += 2`.
- [ ] Test `[8,4,5,0,0,0,0,7]` → `[8,4,5,0,0,0,0,0]`, `[1,0]` → `[1,0]`.
- [ ] Time `O(n)`, Space `O(1)`.